# 05 - Prepare Training/Test Data for Big Screen

## Imports and configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

from config import (
    ANNDATA_BIG_COMBO,
    ANNDATA_BIG_SCREEN_SINGLE_KO,
    ANNDATA_BIG_SCREEN_SINGLE_KO_PER_GENE,
    ANNDATA_LEIDEN_TC_TRANSFERRED,
    GUIDE_MODULES_CSV,
    KO_COLUMNS,
    NEIGHBOR_METRIC,
    N_NEIGHBORS,
    N_PCS,
    N_REGRESS_JOBS,
    PROJECT_DIR,
    REGRESS_OUT_VARS,
)

warnings.filterwarnings("ignore")
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False)

os.chdir(PROJECT_DIR)

## 1. Load combo and big-screen data

In [ ]:
adataCombo = sc.read(ANNDATA_LEIDEN_TC_TRANSFERRED)
adataBigScreen = sc.read(ANNDATA_BIG_SCREEN_SINGLE_KO)

In [ ]:
adataBigFiltered = sc.read(ANNDATA_BIG_SCREEN_SINGLE_KO_PER_GENE)
filteredInGuides = adataBigFiltered.uns["feature_barcode_names_filtered"]
del adataBigFiltered

## 2. Restrict to common genes

In [ ]:
commonGenes = adataBigScreen.var_names[[x in adataCombo.var_names for x in adataBigScreen.var_names]]
adataBigScreen = adataBigScreen[:, commonGenes]
adataCombo = adataCombo[:, commonGenes]

## 3. Big-screen: build cell-category and gene-level KO matrix

In [ ]:
fBarMat = adataBigScreen.obs[filteredInGuides]
fBarMat[fBarMat > 0] = 1
adataBigScreen = adataBigScreen[fBarMat.sum(axis=1) == 1, :]

In [ ]:
adataBigScreen.obs["Cell_category"] = adataBigScreen.obs[filteredInGuides].idxmax(axis=1)
adataBigScreen.obs["Cell_category"] = adataBigScreen.obs["Cell_category"].replace(
    to_replace=[r"ONE_NONGENE_SITE_.*", r"NO_TARGET_.*"],
    value="CONTROL_",
    regex=True,
)
adataBigScreen.obs["Cell_category"] = [
    "_".join(x.split("_")[:-1]) + "_" for x in adataBigScreen.obs["Cell_category"]
]

In [ ]:
feature_barcode_names_filtered_GENES = [
    x for x in filteredInGuides
    if not (x.startswith("ONE_NONGENE_SITE") or x.startswith("NO_TARGET"))
]
fBarMat = adataBigScreen.obs[feature_barcode_names_filtered_GENES]

koGenes = sorted({"_".join(x.split("_")[:-1]) + "_" for x in fBarMat.columns})
grouper = [next(p for p in koGenes if p in c) for c in fBarMat.columns]
koGene = fBarMat.groupby(grouper, axis=1).sum()
koGene[koGene > 0] = 1
koGene.columns = "GENE_" + koGene.columns

adataBigScreen.obs = adataBigScreen.obs.join(koGene, how="inner")

In [ ]:
adataBigScreen.obs["GENE_CONTROL_"] = adataBigScreen.obs["Cell_category"]
adataBigScreen.obs["GENE_CONTROL_"][adataBigScreen.obs["GENE_CONTROL_"] != "CONTROL_"] = 0
adataBigScreen.obs["GENE_CONTROL_"][adataBigScreen.obs["GENE_CONTROL_"] == "CONTROL_"] = 1
adataBigScreen.obs["GENE_CONTROL_"] = adataBigScreen.obs["GENE_CONTROL_"].astype("int64")
adataBigScreen.obs["GENE_CONTROL_"].value_counts()

In [ ]:
adataBigScreen.obs = adataBigScreen.obs.drop(
    adataBigScreen.uns["feature_barcode_names"], axis=1
)

## 4. Restrict big screen to selected guide modules

In [ ]:
guideModules = pd.read_csv(GUIDE_MODULES_CSV, index_col=0)
guideModules.index = "GENE_" + guideModules.GuideName + "_"
guideModules.GuideName = "GENE_" + guideModules.GuideName + "_"

group_gene_list = guideModules.GuideName.tolist() + ["GENE_CONTROL_"]
allGeneCols = [x for x in adataBigScreen.obs.columns if x.startswith("GENE_")]
other_gene_list = [x for x in allGeneCols if x not in group_gene_list]

In [ ]:
fBarMat = adataBigScreen.obs[group_gene_list]
fBarMat[fBarMat > 0] = 1
adataBigScreen = adataBigScreen[fBarMat.sum(axis=1) == 1, :]
adataBigScreen.obs = adataBigScreen.obs.drop(other_gene_list, axis=1)

## 5. Combo screen: rename guide columns and add control indicator

In [ ]:
# Drop double-control cells (NO_TARGET + ONE_NON_GENE_SITE) before the
# obs-column intersection removes the Guide column.
adataCombo = adataCombo[adataCombo.obs["Guide"] != "NO_TARGET-ONE_NON_GENE_SITE", :].copy()

adataCombo.obs = adataCombo.obs.rename(columns={c: f"GENE_{c}_" for c in KO_COLUMNS})

adataCombo.obs["GENE_CONTROL_"] = (
    adataCombo.obs["NO_TARGET"].astype(int) | adataCombo.obs["ONE_NON_GENE_SITE"].astype(int)
)

## 6. Align obs columns across the two datasets

In [ ]:
adataBigScreen.obs["leidenBigScreen"] = adataBigScreen.obs["leiden"]

geneCols = [x for x in adataBigScreen.obs.columns if x.startswith("GENE_")]
for elem in geneCols:
    if elem not in adataCombo.obs.columns:
        adataCombo.obs[elem] = 0

In [ ]:
common_obs_cols = adataBigScreen.obs.columns.intersection(adataCombo.obs.columns)
adataBigScreen.obs = adataBigScreen.obs[common_obs_cols].copy()
adataCombo.obs = adataCombo.obs[common_obs_cols].copy()

## 7. Move counts layer back into X

In [ ]:
adataBigScreen.X = adataBigScreen.layers["counts"].A
del adataBigScreen.layers["counts"]
adataBigScreen.X.shape

In [ ]:
adataCombo.X = adataCombo.layers["counts"].A
adataCombo.X.shape

## 8. Concatenate, normalize, and regress out covariates

In [ ]:
adata = sc.AnnData.concatenate(adataBigScreen, adataCombo)
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=20000)
sc.pp.log1p(adata)

In [ ]:
sc.pp.combat(adata, key="batch")
sc.pp.combat(adata, key="leidenBigScreen")

In [ ]:
REGRESS_OUT_VARS

In [ ]:
sc.pp.regress_out(adata, REGRESS_OUT_VARS, n_jobs=N_REGRESS_JOBS)

## 9. Save merged dataset

In [ ]:
adata.write(ANNDATA_BIG_COMBO)